# PULSE 🩺 — Power BI Report Health Checker

Proof of concept for proactively inspecting Power BI report visuals and surfacing potential failures, blank-data issues, and unusually slow visual exports.

> **Compatibility:** Run this application in Jupyter Classic Notebook 6.x. Its lightweight UI uses the Classic `Jupyter.notebook` frontend API and is not compatible with Notebook 7.

Run all cells once to initialize the interface, then use the green **(Re)Start Run** button for later runs. Review `README.md` before connecting PULSE to a Power BI tenant.

In [ ]:
from IPython.display import HTML, display


def show_app_header_and_button():
    display(HTML("""
        <div style="display:flex;justify-content:center;align-items:center;flex-direction:column;
                    min-height:300px;margin:30px 0;transform:translateX(-50px);">
            <div style="font-family:Arial,sans-serif;font-size:48px;font-weight:bold;color:#4CAF50;
                        display:inline-flex;align-items:center;gap:15px;animation:pulseHeader 2s infinite;">
                PULSE <span style="font-size:48px;">🩺</span>
            </div>
            <div style="font-family:Arial,sans-serif;font-size:18px;color:#555;margin:15px 0 20px;text-align:center;">
                Power BI Report Health Checker
            </div>
            <button onclick="clean_and_run_all()"
                    style="background-color:#4CAF50;color:white;padding:10px 20px;border:none;
                           cursor:pointer;font-size:16px;border-radius:5px;">
                ▶️ (Re)Start Run
            </button>
        </div>
        <style>
            @keyframes pulseHeader {
                0% { transform:scale(1); }
                50% { transform:scale(1.05); }
                100% { transform:scale(1); }
            }
        </style>
        <script>
            $(document).ready(function(){ $('div.input').hide(); });

            function delete_pulse_dynamic_cells() {
                for (var i = Jupyter.notebook.ncells() - 1; i >= 0; i--) {
                    var cell = Jupyter.notebook.get_cell(i);
                    if (cell && cell.metadata && cell.metadata.pulse_dynamic) {
                        Jupyter.notebook.delete_cell(i);
                    }
                }
            }

            function clean_and_run_all() {
                delete_pulse_dynamic_cells();
                window.scrollTo({ top:document.body.scrollHeight, behavior:'smooth' });
                setTimeout(function(){ Jupyter.notebook.execute_cells([1, 2]); }, 300);
            }
        </script>
    """))


show_app_header_and_button()



In [ ]:
import asyncio
import contextlib
import io
import json
import html
import logging
import os
import sys
import time
from datetime import datetime

import ipywidgets as widgets
import nest_asyncio
import requests
from dotenv import load_dotenv
from IPython.display import HTML, Javascript, clear_output, display
from powerbiclient import Report
from powerbiclient.authentication import InteractiveLoginAuthentication

from proxy_support import configure_pac_proxy, restore_proxy_environment
from pulse_helpers import (
    check_for_blanks,
    create_results_workbook,
    is_my_workspace_report,
    make_report_options,
    make_workspace_options,
    parse_skip_visuals,
    should_skip_visual,
)


# ---------- Configuration and logging ----------
load_dotenv()


def env_bool(name, default=False):
    value = os.getenv(name)
    return default if value is None else value.strip().casefold() in {"1", "true", "yes", "on"}


def env_float(name, default, minimum=0.0):
    value = float(os.getenv(name, default))
    if value <= minimum:
        raise ValueError(f"{name} must be greater than {minimum}")
    return value


TENANT_ID = os.getenv("PULSE_TENANT_ID") or None
USE_PAC_DEFAULT = env_bool("PULSE_USE_PAC", False)
REQUEST_TIMEOUT_SECONDS = env_float("PULSE_REQUEST_TIMEOUT_SECONDS", 30)
SLOW_EXPORT_THRESHOLD_SECONDS = env_float("PULSE_SLOW_EXPORT_THRESHOLD_SECONDS", 2.0)
HIGHLIGHT_THRESHOLD_SECONDS = env_float("PULSE_HIGHLIGHT_THRESHOLD_SECONDS", 5.0)
VISUALS_TO_SKIP = parse_skip_visuals(os.getenv("PULSE_SKIP_VISUALS_JSON", "[]"))


def setup_logging():
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(levelname)-8s %(message)s"))
    logging.basicConfig(level=logging.INFO, handlers=[handler], force=True)


setup_logging()
logger = logging.getLogger("pulse")
restore_proxy_environment()

api_proxies = None
int_auth = None
headers = None
workspaces = []
reports = []


# ---------- Persistent UI containers ----------
network_output = widgets.Output()
workspace_output = widgets.Output()
report_output = widgets.Output()
run_button_output = widgets.Output()

display(network_output, workspace_output, report_output, run_button_output)


def clear_dynamic_cells():
    display(Javascript("""
        if (typeof delete_pulse_dynamic_cells === 'function') {
            delete_pulse_dynamic_cells();
        }
    """))


def clear_selectors():
    for output in (network_output, workspace_output, report_output, run_button_output):
        with output:
            clear_output()


# ---------- Power BI REST discovery ----------
def fetch_api_values(url):
    values = []
    next_url = url
    while next_url:
        response = requests.get(
            next_url,
            headers=headers,
            proxies=api_proxies,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        try:
            response.raise_for_status()
        except requests.HTTPError as exc:
            raise RuntimeError(
                f"Power BI REST request failed with HTTP {response.status_code}"
            ) from exc
        payload = response.json()
        values.extend(payload.get("value", []))
        next_url = payload.get("@odata.nextLink")
    return values


def start_powerbi_flow():
    global int_auth, headers, workspaces
    logger.info("Authenticating. A local browser window will open for interactive sign in.\n")
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            int_auth = InteractiveLoginAuthentication(tenant_id=TENANT_ID)
        access_token = int_auth.get_access_token()
    except Exception as exc:
        logger.error(f"❌ Authentication failed: {exc}")
        return

    headers = {"Authorization": f"Bearer {access_token}", "Accept": "application/json"}
    logger.info("✅ Authentication successful.\n")

    # My workspace is not returned by the /groups endpoint. Keep it as a
    # first-class option and add any shared workspaces available to the user.
    workspaces = [{"id": None, "name": "My workspace", "is_personal": True}]
    try:
        shared_workspaces = fetch_api_values("https://api.powerbi.com/v1.0/myorg/groups")
        workspaces.extend(shared_workspaces)
    except (requests.RequestException, ValueError, RuntimeError) as exc:
        logger.warning(f"⚠️ Shared workspace discovery was unavailable: {exc}")

    workspaces.sort(key=lambda workspace: str(workspace.get("name", "")).casefold())
    logger.info(f"✅ Found {len(workspaces)} workspace options.\n")
    show_workspace_selector()


def show_workspace_selector():
    global workspace_select
    with workspace_output:
        clear_output()
        if not workspaces:
            logger.warning("⚠️ No accessible workspaces were returned for this user.")
            return
        workspace_select = widgets.SelectMultiple(
            options=make_workspace_options(workspaces),
            description="Workspaces:",
            layout=widgets.Layout(width="600px", height="200px"),
            style={"description_width": "initial"},
        )
        fetch_button = widgets.Button(description="Fetch Reports")
        fetch_button.on_click(get_selected_workspaces)
        display(widgets.VBox([
            workspace_select,
            widgets.HTML(
                "<div style='font-size:1em;color:#666;margin-top:5px;'>"
                "💡 Use Ctrl/Cmd + Click for multiple items, or Shift + Click for a range."
                "</div>"
            ),
            fetch_button,
        ]))


def get_selected_workspaces(_button):
    global reports
    clear_dynamic_cells()
    with run_button_output:
        clear_output()

    selected = [workspaces[index] for index in workspace_select.value]
    if not selected:
        with report_output:
            clear_output()
            logger.warning("⚠️ Select at least one workspace.")
        return

    reports = []
    for workspace in selected:
        workspace_id = workspace.get("id")
        try:
            if workspace.get("is_personal"):
                workspace_reports = fetch_api_values(
                    "https://api.powerbi.com/v1.0/myorg/reports"
                )
                workspace_reports = [
                    report for report in workspace_reports if is_my_workspace_report(report)
                ]
            else:
                workspace_reports = fetch_api_values(
                    f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/reports"
                )
        except (requests.RequestException, ValueError, RuntimeError) as exc:
            logger.error(
                f"❌ Could not fetch reports for '{workspace.get('name', 'Unnamed workspace')}': {exc}"
            )
            continue
        for report in workspace_reports:
            reports.append({
                "workspace_id": workspace_id,
                "workspace_name": workspace.get("name", "Unnamed workspace"),
                "report_id": report["id"],
                "report_name": report.get("name", "Unnamed report"),
            })

    show_report_selector()


def show_report_selector():
    global report_select
    with report_output:
        clear_output()
        if not reports:
            logger.warning("⚠️ No reports were found in the selected workspaces.")
            return
        report_select = widgets.SelectMultiple(
            options=make_report_options(reports),
            description="Reports:",
            layout=widgets.Layout(width="600px", height="200px"),
            style={"description_width": "initial"},
        )
        confirm_button = widgets.Button(description="Confirm Reports")
        confirm_button.on_click(generate_analysis_code)
        display(
            report_select,
            widgets.HTML(
                "<div style='font-size:1em;color:#666;margin-top:5px;'>"
                "💡 Use Ctrl/Cmd + Click for multiple items, or Shift + Click for a range."
                "</div>"
            ),
            confirm_button,
        )


# ---------- Dynamic Classic Notebook cells ----------
def create_markdown_cell(text):
    display(Javascript(f"""
        var cell = Jupyter.notebook.insert_cell_below('markdown');
        cell.set_text({json.dumps(text)});
        cell.metadata.pulse_dynamic = true;
        var idx = Jupyter.notebook.find_cell_index(cell);
        Jupyter.notebook.execute_cells([idx]);
    """))


def create_hidden_and_execute_cell(code):
    display(Javascript(f"""
        var cell = Jupyter.notebook.insert_cell_below('code');
        cell.set_text({json.dumps(code)});
        cell.metadata.pulse_dynamic = true;
        cell.element.find('div.input').hide();
        var idx = Jupyter.notebook.find_cell_index(cell);
        Jupyter.notebook.execute_cells([idx]);
    """))


def create_analysis_cell(code):
    display(Javascript(f"""
        var cell = Jupyter.notebook.insert_cell_below('code');
        cell.set_text({json.dumps(code)});
        cell.metadata.pulse_dynamic = true;
        window.final_analysis_cell_idx = Jupyter.notebook.find_cell_index(cell);
    """))


def show_run_last_cell_button():
    with run_button_output:
        clear_output()
        display(HTML("""
            <button id="runAnalysisButton" onclick="
                var idx = window.final_analysis_cell_idx;
                if (typeof idx !== 'number' || !Jupyter.notebook.get_cell(idx)) {
                    alert('The generated analysis cell is no longer available. Confirm the reports again.');
                    return;
                }
                Jupyter.notebook.execute_cells([idx]);
                document.getElementById('runAnalysisButton').style.display='none';
                var p = document.getElementById('runAnalysisInstruction');
                if (p) { p.innerHTML='<span style=&quot;font-size:20px;font-weight:bold;&quot;>🛠️ Analysis is running, please wait...</span>'; }
                Jupyter.notebook.get_cell(idx).element.find('div.input').hide();
            " style="background-color:#4CAF50;color:white;padding:10px 20px;border:none;
                     cursor:pointer;font-size:16px;border-radius:5px;margin-top:10px;">
                ▶️ Run Analysis
            </button>
            <p id="runAnalysisInstruction" style="font-size:20px;color:#555;margin-top:10px;">
                <b>Wait for every report to finish rendering before running the analysis.</b><br>
                You can fetch workspaces and reports again if needed.
            </p>
        """))


ANALYSIS_TEMPLATE = r'''
# PULSE generated analysis cell (Classic Notebook only)
nest_asyncio.apply()
clear_selectors()

spinner_html = """
<div id="loadingSpinner" style="position:fixed;top:130px;right:20px;background:#f8f9fa;
     border:2px solid #4CAF50;border-radius:10px;padding:10px 15px;box-shadow:0 4px 8px rgba(0,0,0,.1);
     display:flex;align-items:center;z-index:10000;font:700 20px Arial;color:#4CAF50;">
  <span style="display:inline-block;width:28px;height:28px;border:4px solid #eee;border-top-color:#4CAF50;
        border-radius:50%;animation:pulseSpin 1.2s linear infinite;margin-right:14px;"></span>Analyzing...
</div>
<style>@keyframes pulseSpin { to { transform:rotate(360deg); } }</style>
"""
display(HTML(spinner_html))


VISUALS_PENDING_SENTINEL = [{"name": "__pulse_visuals_pending__"}]


async def fetch_visuals_async(report, page_name):
    # powerbiclient 3.1.1 uses [] as both its pending state and a valid
    # empty-page result. Give this report instance a distinct pending value
    # so the library can observe a frontend response containing [].
    report.PAGE_VISUALS_DEFAULT_STATE = VISUALS_PENDING_SENTINEL
    report._page_visuals = list(VISUALS_PENDING_SENTINEL)
    return await asyncio.to_thread(report.visuals_on_page, page_name)


def fetch_visuals_with_timeout(report, page_name, page_display_name, timeout=20):
    loop = asyncio.get_event_loop()
    try:
        return loop.run_until_complete(
            asyncio.wait_for(fetch_visuals_async(report, page_name), timeout=timeout)
        )
    except asyncio.TimeoutError:
        logger.warning(f"⏳ Timed out while fetching visuals for '{page_display_name}'.")
    except Exception as exc:
        logger.error(f"❌ Could not fetch visuals for '{page_display_name}': {exc}")
    return None


def ensure_page_switch(report, page, max_retries=3, wait_seconds=5):
    page_name = page["name"]
    page_display_name = page["displayName"]
    for attempt in range(1, max_retries + 1):
        logger.info(f"🔄 Attempt {attempt}: setting active page '{page_display_name}'")
        report.set_active_page(page_name)
        time.sleep(wait_seconds)
        visuals = fetch_visuals_with_timeout(report, page_name, page_display_name)
        if visuals is not None:
            return visuals
    logger.warning(f"⚠️ Skipping page after retries: '{page_display_name}'")
    return None


def try_export_visual_data(report, page_name, page_display_name, visual, report_name):
    try:
        exported_data = report.export_visual_data(page_name, visual["name"], rows=13)
        warnings = check_for_blanks(exported_data, visual["type"])
        return {
            "status": "warning" if warnings else "success",
            "criticality": 2 if warnings else 3,
            "report_name": report_name,
            "page_name": page_display_name,
            "visual_title": visual.get("title", "Untitled visual"),
            "visual_type": visual["type"],
            "error_message": "No error",
            "warnings": warnings,
        }
    except Exception as exc:
        return {
            "status": "error",
            "criticality": 1,
            "report_name": report_name,
            "page_name": page_display_name,
            "visual_title": visual.get("title", "Untitled visual"),
            "visual_type": visual.get("type", "Unknown"),
            "error_message": str(exc),
            "warnings": None,
        }


def run_analysis():
    results = []
    skipped_visuals = []
    unsupported_types = {"textbox", "actionbutton", "shape", "basicshape", "image", "bookmarknavigator"}
    report_objects = [__REPORT_VARIABLES__]
    report_names = __REPORT_NAMES__

    logger.info("\n==================================================")
    logger.info("🔎 Starting analysis of selected reports")
    logger.info("==================================================\n")

    for report, report_name in zip(report_objects, report_names):
        logger.info(f"📌 Loading report: {report_name}")
        try:
            pages = report.get_pages()
        except Exception as exc:
            logger.error(f"❌ Could not fetch pages for '{report_name}': {exc}")
            continue

        for page in pages:
            page_display_name = page.get("displayName", page.get("name", "Unnamed page"))
            logger.info(f"📄 Processing page: {page_display_name}")
            try:
                visuals = ensure_page_switch(report, page)
                if visuals is None:
                    logger.warning(f"⚠️ Unresponsive page skipped: '{page_display_name}'")
                    continue
                if len(visuals) == 0:
                    logger.info(f"⏭️ Empty page skipped: '{page_display_name}'")
                    continue

                for visual in visuals:
                    visual_type = visual.get("type", "Unknown")
                    visual_title = visual.get("title", "Untitled visual")
                    if visual_type.casefold() in unsupported_types:
                        logger.info(f"⏭️ Unsupported export type skipped: {visual_title} ({visual_type})")
                        continue
                    if should_skip_visual(
                        report_name, page_display_name, visual_title, VISUALS_TO_SKIP
                    ):
                        skipped_visuals.append({
                            "report_name": report_name,
                            "page_name": page_display_name,
                            "visual_title": visual_title,
                        })
                        logger.info(f"⏭️ Configured visual skipped: {visual_title}")
                        continue

                    time.sleep(2)
                    logger.info(f"✔️ Processing visual: {visual_title} ({visual_type})")
                    started = time.perf_counter()
                    result = try_export_visual_data(
                        report, page["name"], page_display_name, visual, report_name
                    )
                    result["export_time_seconds"] = time.perf_counter() - started
                    results.append(result)

                    if result["status"] == "error":
                        logger.error(f"❌ Export failed for '{visual_title}': {result['error_message']}")
                    elif result["status"] == "warning":
                        logger.warning(f"⚠️ Warning for '{visual_title}': {result['warnings']}")
                    logger.info(
                        f"⏱️ Export completed in {result['export_time_seconds']:.2f} seconds"
                    )
            except Exception as exc:
                logger.error(f"❌ Error while processing '{page_display_name}': {exc}")
    return results, skipped_visuals


analysis_failed = False
try:
    results, skipped_visuals = run_analysis()
except Exception as exc:
    analysis_failed = True
    results, skipped_visuals = [], []
    logger.exception(f"❌ Analysis stopped unexpectedly: {exc}")
finally:
    display(Javascript("var spinner=document.getElementById('loadingSpinner'); if(spinner){spinner.remove();}"))

error_count = sum(result["status"] == "error" for result in results)
warning_count = sum(result["status"] == "warning" for result in results)
success_count = sum(result["status"] == "success" for result in results)
slow_count = sum(
    float(result.get("export_time_seconds") or 0) > SLOW_EXPORT_THRESHOLD_SECONDS
    for result in results
)

file_name = f"PULSE_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
workbook_path = create_results_workbook(
    results,
    skipped_visuals,
    os.path.join(os.getcwd(), "Results", file_name),
    slow_threshold_seconds=SLOW_EXPORT_THRESHOLD_SECONDS,
    highlight_threshold_seconds=HIGHLIGHT_THRESHOLD_SECONDS,
)

if workbook_path:
    display(Javascript("""
        var link=document.createElement('a');
        link.href=Jupyter.notebook.base_url+'files/Results/'+encodeURIComponent(__FILE_NAME__);
        link.download=__FILE_NAME__;
        document.body.appendChild(link); link.click(); link.remove();
    """.replace("__FILE_NAME__", json.dumps(file_name))))
    result_message = f"Results saved to Results/{file_name}"
else:
    result_message = "No workbook was needed because no findings were detected."

if analysis_failed:
    title, accent = "❌ Analysis incomplete", "#dc3545"
elif workbook_path:
    title, accent = "✅ Analysis complete", "#28a745"
else:
    title, accent = "🎯 No issues found", "#28a745"

notification_html = f"""
<div style="position:fixed;top:130px;right:20px;background:#f8f9fa;border-left:5px solid {accent};
            box-shadow:0 4px 8px rgba(0,0,0,.1);padding:15px 20px;border-radius:4px;z-index:1000;
            font-family:Arial,sans-serif;max-width:390px;">
  <div style="font-weight:bold;margin-bottom:10px;">{title}</div>
  <div style="color:#495057;margin-bottom:10px;">{result_message}</div>
  <div style="color:#495057;">{error_count} errors · {warning_count} warnings · {success_count} successful<br>
       {slow_count} slow exports · {len(skipped_visuals)} configured skips</div>
  <button onclick="this.parentNode.remove()" style="position:absolute;top:5px;right:5px;background:none;
          border:none;cursor:pointer;font-size:16px;color:#6c757d;">×</button>
</div>
"""

logger.info("✅ Analysis complete.")
display(Javascript("""
    var headerCell=Jupyter.notebook.get_cell(1);
    if(headerCell){
        headerCell.output_area.append_output({
            output_type:'display_data',
            data:{'text/html':__HTML__, 'text/plain':'PULSE analysis summary'},
            metadata:{}
        });
    }
    setTimeout(function(){
        if(typeof delete_pulse_dynamic_cells==='function'){delete_pulse_dynamic_cells();}
        var appCell=Jupyter.notebook.get_cell(2);
        if(appCell){appCell.clear_output();}
    }, 4000);
""".replace("__HTML__", json.dumps(notification_html))))
'''


def generate_analysis_code(_button):
    clear_dynamic_cells()
    selected_reports = [reports[index] for index in report_select.value]
    if not selected_reports:
        with run_button_output:
            clear_output()
            logger.warning("⚠️ Select at least one report.")
        return

    variable_names = []
    for index, report in enumerate(selected_reports):
        variable_name = f"report_{index}"
        variable_names.append(variable_name)
        create_markdown_cell(f"# 📄 {html.escape(str(report['report_name']))}")
        create_hidden_and_execute_cell(
            f"# Embed selected Power BI report\n"
            f"{variable_name} = Report(group_id={report['workspace_id']!r}, "
            f"report_id={report['report_id']!r}, auth=int_auth)\n"
            f"{variable_name}\n"
        )

    final_code = ANALYSIS_TEMPLATE.replace(
        "__REPORT_VARIABLES__", ", ".join(variable_names)
    ).replace(
        "__REPORT_NAMES__", repr([report["report_name"] for report in selected_reports])
    )
    create_analysis_cell(final_code)
    show_run_last_cell_button()


# ---------- Network selection and start ----------
def setup_network_selector():
    with network_output:
        clear_output()
        options = [
            ("🌐 Standard connection", False),
            ("🧭 Automatic PAC proxy", True),
        ]
        selector = widgets.Dropdown(
            options=options,
            value=USE_PAC_DEFAULT,
            description="Network mode:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="450px"),
        )
        button = widgets.Button(description="Apply and Start")
        status = widgets.Output()

        def apply_and_start(_button):
            global api_proxies
            with status:
                clear_output()
                restore_proxy_environment()
                api_proxies = None
                if selector.value:
                    try:
                        api_proxies = configure_pac_proxy(
                            timeout_seconds=REQUEST_TIMEOUT_SECONDS
                        )
                        logger.info("🌐 PAC proxy configured successfully.\n")
                    except RuntimeError as exc:
                        logger.error(f"❌ PAC proxy setup failed: {exc}")
                        logger.info("Choose Standard connection to continue without PAC discovery.")
                        return
                else:
                    logger.info("🌐 Using the system's standard network configuration.\n")

            clear_dynamic_cells()
            for output in (workspace_output, report_output, run_button_output):
                with output:
                    clear_output()
            start_powerbi_flow()

        button.on_click(apply_and_start)
        display(HTML("<h4>📡 Choose a network mode:</h4>"), widgets.VBox([selector, button, status]))


setup_network_selector()

